## Instructions

1. Do not write your name on the assignment. Be careful about any warnings that might display some file path with your name included.

2. You may talk to a friend, discuss the questions and potential directions for solving them. However, you need to write your own solutions and code separately, and not as a group activity.

3. You are expected to carefully read and comply with the [generative AI policy](https://canvas.northwestern.edu/courses/233157/pages/generative-ai-policy?module_item_id=3421526) of this course.

4. Write your code in the *Code* cells and print the instructed output. If you are instructed to explain something in words, write your answer in the *Markdown* cells of the Jupyter notebook. Ensure that the solution is coded and/or written neatly enough to understand and grade.

5. Use [Quarto](https://quarto.org/docs/output-formats/html-basics.html) to render the *.ipynb* file as HTML. You will need to open the command prompt, navigate to the directory containing the file, and use the command: `quarto render filename.ipynb --to html`. Submit the HTML file.

6. This assignment is worth 100 points and is due on **May 31, 2025 at 11:59 pm**. 

7. **Five points given for properly formatting the assignment**. The breakdown is as follows:
- The submission must be an HTML file rendered using Quarto. (1 point).
- Your name should not be visible in the HTML file (including the file path in any warning that your code returns). (1 point)
- There are not excessively long outputs of extraneous information. (e.g. no printouts of entire data frames without good reason; there are not long printouts of which iteration a loop is on; there are not long sections of commented-out code, etc.) (1 point)
- Final answers for each question are written in Markdown cells. (1 point).
- There is no piece of unnecessary / redundant code, and no unnecessary / redundant text. (1 point)

## 1) AdaBoost vs Random Forest (4 points)

Which model among AdaBoost and Random Forest is more sensitive to outliers? **(1 point)** Explain your reasoning using the theory you learned about the training process of both models. **(3 points)**

## 2) Regression with Boosting (50 points)

In this question, you will use the **miami_housing.csv** file. You can find the description for the variables [here](https://www.kaggle.com/datasets/deepcontractor/miami-housing-dataset).

The `SALE_PRC` variable is the regression response and the rest of the variables, except `PARCELNO`, are the predictors.

### a)

Read the dataset. Create the training and test sets with a 60%-40% split and `random_state = 1`. **(1 point)**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, cross_val_predict, cross_validate
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold, StratifiedKFold, KFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

In [3]:
miami_data = pd.read_csv("miami-housing.csv")
predictors = miami_data.drop(columns = ["SALE_PRC", "PARCELNO"], axis=1)
target = miami_data["SALE_PRC"]
X_train, X_test, y_train, y_test = train_test_split(predictors, target, test_size=0.4, random_state=1)

### b)

Tune an AdaBoost regressor to get below a **5-fold** cross-validation MAE of $48,000. The requirements are:

-  Keep the `random_state` of **any object that takes a `random_state` input** as 1.
-  Do not use any loops while tuning the model.
-  In your final submission, include an actual grid that is run with **at least 10** different hyperparameter combinations. 

Any solution that fails to meet these requirements will not receive any credit. Note that any other aspects are entirely up to you, including the grid values, how to approach any coarse and/or fine grids, and the cross-validation settings (except the number of folds given above).

**(10 points)**

In [4]:
base_model = DecisionTreeRegressor(random_state = 1)

model = AdaBoostRegressor(
    random_state = 1,
    estimator = base_model
)

grid = {
    "estimator__max_depth": [19,20],
    "n_estimators": range(126, 129),
    "learning_rate": [0.1, 1]
}

gscv = GridSearchCV(
    model,
    grid,
    scoring = "neg_mean_absolute_error",
    cv = KFold(n_splits=5, random_state=1, shuffle=True),
    n_jobs = int(os.getenv("SLURM_NPROCS", 1)),
    verbose = 1
)

gscv.fit(X_train, y_train)
print("best CV score: ", -gscv.best_score_)
print(gscv.best_params_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
best CV score:  47896.90629635469
{'estimator__max_depth': 20, 'learning_rate': 1, 'n_estimators': 127}


### c)

Find the test MAE of the tuned AdaBoost model in Part b to see if it generalizes well. **(1 point)** 

In [5]:
gscv_model = gscv.best_estimator_
y_pred_gscv = gscv_model.predict(X_test)

test_mae = mean_absolute_error(y_test, y_pred_gscv)
print(test_mae)

44708.14507607808


### d)

Using the tuned AdaBoost model in Part b, print the predictor names in **decreasing order** of importance. You need to print a DataFrame with the predictor names in the first column and the importance values in the second. **(1 point)**

**Note:** Feature importances can be found using the same line of code for all of the models in this assignment. It is asked only for the AdaBoost model and omitted from the remaining models to avoid repetition.

In [6]:
feature_imps = pd.DataFrame({
    "feature": X_train.columns,
    "importance": gscv_model.feature_importances_
}).sort_values(by="importance", ascending = False)

print(feature_imps)

              feature  importance
3        TOT_LVG_AREA    0.351399
6          OCEAN_DIST    0.166532
8           CNTR_DIST    0.077141
1           LONGITUDE    0.070799
14  structure_quality    0.063169
7          WATER_DIST    0.061976
4       SPEC_FEAT_VAL    0.048220
5           RAIL_DIST    0.036866
9          SUBCNTR_DI    0.032741
2          LND_SQFOOT    0.028853
11                age    0.023743
10           HWY_DIST    0.017007
0            LATITUDE    0.015697
13         month_sold    0.005693
12         avno60plus    0.000164


### e)

Moving on to Gradient Boosting, which hyperparameter should be kept out of the grid search? **(1 point)** To what input value should it be fixed to? **(1 point)** What are the advantages of that input value over other possible input values? **(2 points)**

We should keep the loss hyperparameter out of the grid search for Gradient Boosting and instead fix it to huber. Huber is preferred to MSE and MAE because it functions like MSE for smaller residuals and like MAE for larger residuals, reducing the importance of outliers and avoids overfitting. 

### f)

Tune a Gradient Boosting regressor to get below a **5-fold** cross-validation MAE of $45,000. The requirements are:

-  Keep the `random_state` of **any object that takes a `random_state` input** as 1.
-  Do not use any loops while tuning the model.
-  In your final submission, include an actual grid that is run with **at least 10** different hyperparameter combinations. 

Any solution that fails to meet these requirements will not receive any credit. Note that any other aspects are entirely up to you, including the grid values, how to approach any coarse and/or fine grids, and the cross-validation settings (except the number of folds given above).

**(10 points)**

In [9]:
gbm_reg = GradientBoostingRegressor(random_state = 1, loss = "huber")

grid_gbm = {
    'n_estimators': [1730, 1750],
    'max_depth': [4],
    'learning_rate': [0.01, 0.01, 0.1],
    'subsample': [0.75, 1]
}

gscv_gbm = GridSearchCV(
    gbm_reg,
    grid_gbm,
    scoring = "neg_mean_absolute_error",
    cv = 5,
    n_jobs = int(os.getenv("SLURM_NPROCS", 1)),
    verbose = 1
)

gscv_gbm.fit(X_train, y_train)
print("best CV score: ", -gscv_gbm.best_score_)
print(gscv_gbm.best_params_)


Fitting 5 folds for each of 12 candidates, totalling 60 fits
best CV score:  44997.13223570492
{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 1750, 'subsample': 0.75}


### g)

Find the test MAE of the tuned Gradient Boosting model in Part f to see if it generalizes well. **(1 point)**

In [10]:
best_gbm_model = gscv_gbm.best_estimator_
y_pred_gbm = best_gbm_model.predict(X_test)
print(mean_absolute_error(y_test, y_pred_gbm))


44012.523771613996


### h)

Explain how the tuned hyperparameters of AdaBoost and Gradient Boosting affect the bias and the variance of their model. Note that most hyperparameters are the same across these two models, so you only need one explanation to cover both. (Make sure to include four hyperparameters in total.) **(4 points)**

- increasing n_estimators decreases bias by including more weak learners but also increases variance as it risks memorizing the noise in the training data. Vice versa for decreasing n_estimators
- increasing max_depth decreases the bias of both models by increasing the complexity of the base models. Increasing max_depth also increases variance because letting base models get too deep risks memorizing noise in training data
- increasing learning_rate decreases bias by more abruptly changing how the next tree in the sequence is trained for Adaboost and increasing the contribution of the next tree to the prediction for Gradient Boosting. however, increasing learning_rate could increase variance by overfitting the training data
- increasing subsample decreases bias by showing each weak learner more of the training data but could also increase variance by making the weak learners highly correlated

### i)

Moving on to XGBoost: 

- What are the additions that make XGBoost superior to Gradient Boosting? You need to explain this in terms of runtime **(1 point)** with its reason **(2 points)** and the hyperparameters **(1 point)** with their effect on the model behavior. **(2 points)**.

- What is missing from XGBoost that is well-implemented in Gradient Boosting? **(1 point)**

XGBoost has faster runtime than Gradient Boosting because it processes the nodes of each tree in parallel. It also uses histograms of features to more quickly arrive at accurate decision rules. XGBoost also includes hyperparameters reg_lambda (penalizes large leaf weights, shrinks them), gamma (prunes unnecessary leaves), and the colsample hyperparameters (determines feature subset each tree/level/node sees) to reduce overfitting. However, huber loss is missing from XGBoost, which is well implemented in Gradient Boosting.

### j)

Tune an XGBoost regressor to get below a **5-fold** cross-validation MAE of $43,500. The requirements are:

-  Keep the `random_state` of **any object that takes a `random_state` input** as 1.
-  Do not use any loops while tuning the model.
-  In your final submission, include an actual grid that is run with **at least 10** different hyperparameter combinations. 

Any solution that fails to meet these requirements will not receive any credit. Note that any other aspects are entirely up to you, including the grid values, how to approach any coarse and/or fine grids, and the cross-validation settings (except the number of folds given above).

**(10 points)**

In [10]:
from xgboost import XGBRegressor, XGBClassifier
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


xgb_model = XGBRegressor(
    random_state = 12,
    objective = 'reg:squarederror'
)

xgb_grid = {
    'max_depth': [4,6,8], 
    'n_estimators': [2000, 2500, 3000], 
    'learning_rate': [0.01, 0.1], 
    'subsample': [1], 
    "reg_lambda": [0.01],
    "gamma": [0], 
    'colsample_bytree': [0.25, 0.5]
}

gscv_xgb = GridSearchCV(
    xgb_model,
    xgb_grid,
    scoring = "neg_mean_absolute_error",
    cv = KFold(n_splits=5, random_state=1, shuffle=True),
    n_jobs = int(os.getenv("SLURM_NPROCS", 1)),
    verbose = 1
)

gscv_xgb.fit(X_train, y_train)
print("best CV score:", -gscv_xgb.best_score_)
print(gscv_xgb.best_params_)


/projects/e32107/src/stats-303/env2/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


Fitting 5 folds for each of 36 candidates, totalling 180 fits


/projects/e32107/src/stats-303/env2/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/projects/e32107/src/stats-303/env2/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux201

best CV score: 43299.843522722906
{'colsample_bytree': 0.5, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 3000, 'reg_lambda': 0.01, 'subsample': 1}


### k)

Find the test MAE of the tuned XGBoost model in Part j to see if it generalizes well. **(1 point)**

In [11]:
best_xgb_model = gscv_xgb.best_estimator_
y_pred_xgb = best_xgb_model.predict(X_test)
print(mean_absolute_error(y_test, y_pred_xgb))

41332.80211651041


## 2) Classification with Boosting (41 points)

In this question, you will use the **train.csv** and **test.csv** files. Each observation is a marketing call from a banking institution. The `y` variable is the classification response and represents whether the client subscribed for a term deposit (1) or not (0).

The predictors are `age`, `day`, `month`, and `education`.

### a)

Preprocess the data:

- Read the files. Create the training and the test datasets.
- Convert the response to 1s and 0s.
- One-hot-encode the categorical predictors (**Do not use `drop_first`**).

**(1 point)**

In [12]:
from sklearn.preprocessing import OneHotEncoder


train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train["y"] = train["y"].apply(lambda x: 1 if x == "yes" else 0)
test["y"] = test["y"].apply(lambda x: 1 if x == "yes" else 0)

X_train = train.drop(columns=["y"], axis=1)
y_train = train["y"]
X_test = test.drop(columns=["y"], axis=1)
y_test = test["y"]


categorical_vars = ["education", "month"]

X_train_numerical = X_train.drop(columns=categorical_vars, axis=1)
X_test_numerical = X_test.drop(columns=categorical_vars, axis=1)

encoder = OneHotEncoder(sparse_output=False)
X_train_encoded = encoder.fit_transform(X_train[categorical_vars])
X_test_encoded = encoder.transform(X_test[categorical_vars])

X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out(categorical_vars))
X_test_encoded = pd.DataFrame(X_test_encoded, columns=encoder.get_feature_names_out(categorical_vars))
X_train_cleaned = pd.concat([X_train_numerical, X_train_encoded], axis=1)
X_test_cleaned = pd.concat([X_test_numerical, X_test_encoded], axis=1)


### b)

Moving on to LightGBM and CatBoost, what are their advantages compared to Gradient Boosting and XGBoost? **(2 points)** How are these advantages implemented into the models? **(2 points)** Do any of them have any disadvantages? If there are any, describe them. **(1 point)**

LightGBM runs faster than XGBoost and Gradient Boosting because it excludes observations with small contributions to the cost function and bundles sparse predictors into one. Catboost is much better than XGBoost and Gradient Boosting at avoiding overfitting, since it trains each tree and calculates the cost on different subsets of the training data. It also more efficiently handles categorical variables. LightGBM may sometimes be less reliable than XGBoost and Gradient Boosting, a possible disadvantage.

### c)

In all extensions of Gradient Boosting, (XGBoost, LightGBM and CatBoost) is there an additional hyperparameter you can use to handle a certain issue that is specific to classification? **(1 point)** If yes, describe what it stands for **(1 point)** and how its value should be handled most efficiently. **(1 point)**

This additional hyperparameter is scale_pos_weight, which assigns higher weights to Class 1 observations, ensuring that Class 1 observations are more important to predict. This value should not be tuned and should instead be fixed in the model. A good rule of thumb is to fix it to the proportion of Class 0 to Class 1 observations found in the data.

### d)

Tune a LightGBM classifier to get above a **5-fold** cross-validation accuracy of 70% **and** a **5-fold** cross-validation recall of 65%. The requirements are:

-  Keep the `random_state` of **any object that takes a `random_state` input** as 1.
-  Do not use any loops while tuning the model.
-  In your final submission, include an actual grid that is run with **at least 10** different hyperparameter combinations. 

Any solution that fails to meet these requirements will not receive any credit. Note that any other aspects are entirely up to you, including the grid values, how to approach any coarse and/or fine grids, and the cross-validation settings (except the number of folds given above).

**Hint:** Remember to tune the decision threshold as well for classifiers.

**(15 points)**

In [15]:
from lightgbm import LGBMRegressor, LGBMClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

lgbm_model = LGBMClassifier(
    random_state = 1,
    num_threads = 1,
    scale_pos_weight = 7.53, # ratio between number of 0s and 1s in y_train
    verbosity = -1
)

lgbm_grid = {
    'max_depth': [4,6,8], 
    'n_estimators': [2000, 2500, 3000], 
    'learning_rate': [0.01, 0.1], 
    "subsample": [0.6, 0.8,1],
    
    "reg_lambda": [0.01, 0.1, 1],
    'colsample_bytree': [0.25, 0.5]
}

gscv_lgbm = GridSearchCV(
    lgbm_model,
    lgbm_grid,
    scoring=["accuracy", "recall"],
    refit="accuracy",
    cv=cv,
    n_jobs=int(os.getenv("SLURM_NPROCS", 1)),
    verbose=0
)

gscv_lgbm.fit(X_train_cleaned, y_train)
print(gscv_lgbm.best_params_)
print(gscv_lgbm.best_score_)

{'colsample_bytree': 0.5, 'learning_rate': 0.01, 'max_depth': 8, 'n_estimators': 2500, 'reg_lambda': 1, 'subsample': 0.6}
0.7917714285714286


In [18]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import recall_score, accuracy_score

thresholds = np.arange(0, 1.01, 0.01)

tuned_clf = gscv_lgbm.best_estimator_
y_pred_probs = cross_val_predict(tuned_clf, X_train_cleaned, y_train, cv=cv, method='predict_proba')[:, 1]
cv_results = pd.DataFrame(columns=['Threshold', 'Recall', 'Accuracy'])


counter = 0

for thr in thresholds:
    cv_results.loc[counter, "Threshold"] = thr
    cv_results.loc[counter, "Recall"] = recall_score(y_train, y_pred_probs > thr)
    cv_results.loc[counter, "Accuracy"] = accuracy_score(y_train, y_pred_probs > thr)
    counter += 1

cv_results[(cv_results["Recall"] >= 0.65) & (cv_results["Accuracy"] >= 0.7)]

,Threshold,Recall,Accuracy
43,0.43,0.659113,0.710343


In [19]:
tuned_rows = cv_results[(cv_results["Recall"] >= 0.65) & (cv_results["Accuracy"] >= 0.7)]
tuned_thr = tuned_rows["Threshold"].values[0]
print(tuned_thr)

0.43


### e)

Find the test accuracy and test recall of the tuned LightGBM model and the tuned threshold in Part d to see if they generalize well. **(1 point)**

In [20]:
y_pred_proba_test = tuned_clf.predict_proba(X_test_cleaned)[:, 1]
y_pred_test = y_pred_proba_test > tuned_thr
print(recall_score(y_test, y_pred_test))
print(accuracy_score(y_test, y_pred_test))

0.6795952782462057
0.7131676775582078


### f)

Tune a CatBoost classifier to get above a **5-fold** cross-validation accuracy of 70% **and** a **5-fold** cross-validation recall of 65%. The requirements are:

-  Keep the `random_state` of **any object that takes a `random_state` input** as 1.
-  Do not use any loops while tuning the model.
-  In your final submission, include an actual grid that is run with **at least 10** different hyperparameter combinations. 

Any solution that fails to meet these requirements will not receive any credit. Note that any other aspects are entirely up to you, including the grid values, how to approach any coarse and/or fine grids, and the cross-validation settings (except the number of folds given above).

**Hint:** Remember to tune the decision threshold as well for classifiers.

**(15 points)**

In [24]:
from catboost import CatBoostRegressor, CatBoostClassifier

cb_model = CatBoostClassifier(
    random_state = 1,
    thread_count = 1,
    scale_pos_weight = 7.53, # ratio between number of 0s and 1s in y_train
    verbose = False
)

cb_grid = {
    'max_depth': [4,6,8], 
    'n_estimators': [100, 200, 300], 
    'learning_rate': [0.01, 0.1], 
    "subsample": [0.8,1],
    
    "reg_lambda": [0.1, 1],
}

cb_gscv = GridSearchCV(
    cb_model,
    cb_grid,
    cv = cv,
    scoring=["accuracy", "recall"],
    refit="accuracy",
    n_jobs=int(os.getenv("SLURM_NPROCS", 1)),
    verbose=0
)

cb_gscv.fit(X_train_cleaned, y_train)

print("best score:", cb_gscv.best_score_)
print(cb_gscv.best_params_)



/projects/e32107/src/stats-303/env2/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/projects/e32107/src/stats-303/env2/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux201

best score: 0.7906285714285713
{'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'reg_lambda': 1, 'subsample': 1}


In [25]:
tuned_clf_cat = cb_gscv.best_estimator_
y_pred_probs_cat = cross_val_predict(tuned_clf, X_train_cleaned, y_train, cv=cv, method='predict_proba')[:, 1]
cv_results_cat = pd.DataFrame(columns=['Threshold', 'Recall', 'Accuracy'])


counter = 0

for thr in thresholds:
    cv_results_cat.loc[counter, "Threshold"] = thr
    cv_results_cat.loc[counter, "Recall"] = recall_score(y_train, y_pred_probs_cat > thr)
    cv_results_cat.loc[counter, "Accuracy"] = accuracy_score(y_train, y_pred_probs_cat > thr)
    counter += 1

cv_results_cat[(cv_results_cat["Recall"] >= 0.65) & (cv_results_cat["Accuracy"] >= 0.7)]

,Threshold,Recall,Accuracy
43,0.43,0.659113,0.710343


In [26]:
tuned_rows_cat = cv_results_cat[(cv_results_cat["Recall"] >= 0.65) & (cv_results_cat["Accuracy"] >= 0.7)]
tuned_thr_cat = tuned_rows_cat["Threshold"].values[0]
print(tuned_thr_cat)

0.43


### g)

Find the test accuracy and test recall of the tuned CatBoost model and the tuned threshold in Part f to see if they generalize well. **(1 point)**

In [31]:
y_pred_proba_cat = tuned_clf_cat.predict_proba(X_test_cleaned)[:, 1]
y_pred_cat = y_pred_proba_cat > tuned_thr_cat
print(recall_score(y_test,y_pred_cat))
print(accuracy_score(y_test,y_pred_cat))

0.6795952782462057
0.716298180395226
